In [1]:
import os
import sys
import glob

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from pycaret.clustering import setup, create_model, assign_model, models, pull
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from datetime import datetime
import warnings
import time

from utils import preprocessing

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
warnings.filterwarnings('ignore')
DATA_PATH = '../data'
OUTPUT_DIR = '../results'
IMAGE_DIR = '../images'
    
document_df = preprocessing.get_default_data()

📂 51개 파일 발견
🔄 텍스트 전처리 시작...
   옵션: HTML제거=True, URL제거=True, 숫자제거=True
   옵션: 불용어제거=True, Lemmatization=True, Stemming=False
✅ 전처리 완료:
   - 원본 문서 수: 51
   - 제거된 빈 문서: 0
   - 최종 문서 수: 51
   - 평균 단어 수: 1266.9


In [ ]:
# TF-IDF > SVD(차원 축소) > KMeans 여러값 평가 > 
# Silhouette/Davies-Bouldin/Calinski-Harabasz 지표 계산 > 
# 최적 k 자동 선택(지표 정규화 및 종합 점수 계산)

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

def auto_kmeans_clustering(document_df, text_column='processed_text', max_features=5000, svd_dim=50, k_range=(2,10)):
    """
    document_df만 넣으면 자동으로 군집화 최적 k를 선택하는 함수
    
    Parameters:
        document_df (pd.DataFrame): 텍스트 데이터프레임
        text_column (str): 텍스트 컬럼명 (기본값: 'processed_text')
        max_features (int): TF-IDF 최대 특성 수
        svd_dim (int): SVD 차원 축소 크기
        k_range (tuple): KMeans k 탐색 범위 (기본: 2~10)
    
    Returns:
        dict: 각 k별 지표와 최적 k 결과
    """
    # 1. TF-IDF 벡터화
    texts = document_df[text_column]
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
    X = vectorizer.fit_transform(texts)

    # 2. 차원 축소 (SVD)
    svd = TruncatedSVD(n_components=svd_dim, random_state=42)
    X_reduced = svd.fit_transform(X)

    # 3. 여러 k 값 평가
    scores = {}
    for k in range(k_range[0], k_range[1] + 1):
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(X_reduced)
        
        if len(set(labels)) > 1:
            sil = silhouette_score(X_reduced, labels)
            db = davies_bouldin_score(X_reduced, labels)
            ch = calinski_harabasz_score(X_reduced, labels)
            scores[k] = {"silhouette": sil, "davies_bouldin": db, "calinski": ch}
        else:
            scores[k] = {"silhouette": None, "davies_bouldin": None, "calinski": None}

    # 4. 지표 정규화 및 최적 k 선택
    sil_values = [v["silhouette"] for v in scores.values() if v["silhouette"] is not None]
    db_values  = [v["davies_bouldin"] for v in scores.values() if v["davies_bouldin"] is not None]
    ch_values  = [v["calinski"] for v in scores.values() if v["calinski"] is not None]

    # 정규화 함수
    def normalize(values, higher_is_better=True):
        arr = np.array(values)
        if higher_is_better:
            return (arr - arr.min()) / (arr.max() - arr.min())
        else:
            return (arr.max() - arr) / (arr.max() - arr.min())

    sil_norm = normalize(sil_values, higher_is_better=True)
    db_norm  = normalize(db_values, higher_is_better=False)
    ch_norm  = normalize(ch_values, higher_is_better=True)

    # 종합 점수 계산
    combined_scores = sil_norm + db_norm + ch_norm
    
    # 최적 k 선택
    valid_keys = [k for k,v in scores.items() if v["silhouette"] is not None]
    best_index = np.argmax(combined_scores)
    best_k = valid_keys[best_index]

    return {
        "scores": scores,
        "best_k": best_k,
        "best_metrics": scores[best_k],
        "combined_score": combined_scores[best_index]
    }

In [ ]:
result = auto_kmeans_clustering(document_df)

print("=== 최적 k 선택 결과 ===")
print(f"Best k = {result['best_k']}")
print(f"Silhouette = {result['best_metrics']['silhouette']:.3f}")
print(f"Davies-Bouldin = {result['best_metrics']['davies_bouldin']:.3f}")
print(f"Calinski-Harabasz = {result['best_metrics']['calinski']:.3f}")
print(f"Combined Score = {result['combined_score']:.3f}")

result_bestK = '''
=== 최적 k 선택 결과 ===
Best k = 10
Silhouette = 0.152
Davies-Bouldin = 1.789
Calinski-Harabasz = 2.764
Combined Score = 2.000

======== 설 명 =========
- Silhouette: 높을수록 좋음
- Calinski-Harabasz: 높을수록 좋음
- Davies-Bouldin: 낮을수록 좋음
- 각 지표를 0~1 범위로 정규화한 뒤 합산 → 종합 점수 계산
- 종합 점수가 가장 높은 k를 최적 k로 선택
'''

=== 최적 k 선택 결과 ===
Best k = 10
Silhouette = 0.152
Davies-Bouldin = 1.789
Calinski-Harabasz = 2.764
Combined Score = 2.000
